In [ ]:
import importlib
import sys

# Force reload of helpers module to pick up latest changes
if 'helpers' in sys.modules:
    importlib.reload(sys.modules['helpers'])
    importlib.reload(sys.modules['helpers.database'])
    importlib.reload(sys.modules['helpers.logging_config'])

In [0]:
# ==============================================================================
# CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================

from helpers import create_connection, load_bronze_table, write_gold_table, setup_logger
from pyspark.sql.functions import xxhash64, col, datediff, when
import pyspark.sql.functions as F
import time

# Setup logging
logger = setup_logger("load_facts")

# Initialize connection
c = create_connection(spark, dbutils)
logger.info("=" * 70)
logger.info("FACT LOAD JOB STARTED")
logger.info("=" * 70)


In [0]:
# ==============================================================================
# BRONZE LAYER: Load source tables (in-memory DataFrames)
# ==============================================================================

service_bronze = load_bronze_table(c, "service")
rental_bronze = load_bronze_table(c, "rental")
staff_bronze = load_bronze_table(c, "staff")
inventory_bronze = load_bronze_table(c, "inventory")
payment_bronze = load_bronze_table(c, "payment")


In [0]:
# ==============================================================================
# SILVER LAYER: Clean and prepare DataFrames
# ==============================================================================

logger.info("SILVER: Preparing cleaned DataFrames")

# Pass-through tables (no cleaning needed)
service_silver = service_bronze

# Build denormalized rental view with all necessary joins
# This is data preparation, not business logic, so belongs in silver
logger.info("SILVER: Building denormalized rental view")

# Join rental with staff to get store_id
rental_with_staff = rental_bronze.alias("rental").join(
    staff_bronze.select(
        col("staff_id"),
        col("store_id")
    ).alias("staff"),
    col("rental.staff_id") == col("staff.staff_id"),
    "left"
).select(
    col("rental.*"),
    col("staff.store_id")
)

# Join with inventory to get car_id
rental_with_inventory = rental_with_staff.alias("rental_staff").join(
    inventory_bronze.select("inventory_id", "car_id").alias("inv"),
    col("rental_staff.inventory_id") == col("inv.inventory_id"),
    "left"
).select(
    col("rental_staff.*"),
    col("inv.car_id")
)

# Join with payment
rental_silver = rental_with_inventory.alias("rental_inv").join(
    payment_bronze.select(
        col("rental_id"),
        col("payment_date"),
        col("amount").alias("payment_amount")
    ).alias("payment"),
    col("rental_inv.rental_id") == col("payment.rental_id"),
    "left"
).select(
    col("rental_inv.*"),
    col("payment.payment_date"),
    col("payment.payment_amount")
)

logger.info("SILVER: Data preparation complete")


In [0]:
# ==============================================================================
# GOLD: FACT_SERVICE
# ==============================================================================

logger.info("GOLD: Building fact_service")
start_time = time.time()

fact_service = service_silver.select(
    "service_id",
    "service_date",
    "service_type",
    "service_cost",
    "inventory_id",
).withColumn(
    "service_key", xxhash64(col("service_id"))
).withColumn(
    "car_key", xxhash64(col("inventory_id"))
).drop(
    "inventory_id"
)

write_gold_table(fact_service, "fact_service", mode="overwrite", partition_by=["service_date"])
logger.info(f"GOLD: fact_service completed in {time.time() - start_time:.2f}s")


In [ ]:
# ==============================================================================
# GOLD: FACT_RENTAL
# ==============================================================================

logger.info("GOLD: Building fact_rental")
start_time = time.time()

# Build fact table with surrogate keys and calculated columns
# rental_silver already has all joins from silver layer
fact_rental = (
    rental_silver.select(
        "rental_id",
        "rental_rate",
        "payment_amount",
        "customer_id",
        "car_id",
        "staff_id",
        "store_id",
        "rental_date",
        "return_date",
        "payment_date",
        "payment_deadline"
    )
    # Add surrogate keys
    .withColumn("rental_key", xxhash64(col("rental_id")))
    .withColumn("customer_key", xxhash64(col("customer_id")))
    .withColumn("car_key", xxhash64(col("car_id")))
    .withColumn("staff_key", xxhash64(col("staff_id")))
    .withColumn("store_key", xxhash64(col("store_id")))

    # Add date surrogate keys (handle nulls)
    .withColumn("rental_date_key", xxhash64(col("rental_date")))
    .withColumn("return_date_key",
        F.when(col("return_date").isNotNull(), xxhash64(col("return_date"))).otherwise(None))
    .withColumn("payment_date_key",
        F.when(col("payment_date").isNotNull(), xxhash64(col("payment_date"))).otherwise(None))
    .withColumn("payment_deadline_date_key", xxhash64(col("payment_deadline")))

    # Add calculated business metrics
    .withColumn(
        "rental_amount",
        col("rental_rate") * datediff(col("return_date"), col("rental_date"))
    )
    .withColumn("rental_duration", datediff(col("return_date"), col("rental_date")))
    .withColumn(
        "payment_delay_duration",
        datediff(col("payment_date"), col("payment_deadline"))
    )

    # Drop business keys and intermediate columns (keep only surrogate keys)
    .drop(
        "customer_id",
        "car_id",
        "staff_id",
        "store_id",
        "return_date",
        "payment_date",
        "payment_deadline"
    )
)

write_gold_table(fact_rental, "fact_rental", mode="overwrite", partition_by=["rental_date"])
logger.info(f"GOLD: fact_rental completed in {time.time() - start_time:.2f}s")

# ==============================================================================
# JOB COMPLETION
# ==============================================================================
logger.info("=" * 70)
logger.info("FACT LOAD JOB COMPLETED SUCCESSFULLY")
logger.info("=" * 70)
